# Module 3: Sparse vs Dense vs Hybrid Search

## What you will do

1. See the gap that dense-only search leaves on exact names and codes.
2. Compare the two families of search, dense (meaning) and sparse (exact tokens).
3. Build a hybrid collection and fuse both by rank.
4. Run the dense vs hybrid experiment and watch the ranking change.
5. Prove where a filter has to go in a hybrid query, by breaking it on purpose.

**Tip for Colab:** run cells top to bottom. The first install downloads small embedding models, so it takes a minute.

Companion notebook to the [Module 3 lesson](https://qdrant.tech/course/beginners/module-3/).

## Setup

`qdrant-client[fastembed]` bundles local embedding models. Passing a `models.Document` lets the client embed text for us before upload and at query time, so we never manage the vectors by hand.

In [ ]:
!pip install -q "qdrant-client[fastembed]" 

## 1. Where We Left Off

In Module 2 you built a full pipeline: raw text to vector to store to top-K query. Dense-only retrieval is great for semantic and contextual search, but it struggles on precise product names and model numbers.

Take the query `iPhone 15`. The user wants exactly this product: no synonyms, no paraphrasing. Dense-only search tends to return the whole product line, because "iPhone 14", "iPhone 15", and "iPhone 15 Pro Max" sit close together in embedding space. IDs, codes, and specific model names need exact matching, not semantic neighborhood. That is the gap sparse search fills, and we will reproduce it live in Section 4.

## 2. The Two Families of Search

### Dense search (semantic)

A dense vector has a small, fixed number of dimensions (for example 384), and every dimension holds a value. Two texts with similar meaning produce vectors that are close in space, even if they share no words. That is why `car repair` sits near `automobile maintenance`.

### Sparse search (keyword based)

Sparse vectors are token based. Each dimension maps to a token, and only the tokens that actually appear carry a non-zero value. A vocabulary can be tens of thousands of tokens, but a given text activates only the handful it contains, so sparse vectors are stored as two parallel arrays: the `indices` of the non-zero dimensions and the `values` at those positions.

```python
sparse_vector = {
    "indices": [142, 9325, 44001],  # token IDs: 'nike', 'pegasus', '40'
    "values":  [2.3,  1.2,   0.8],  # weight per token
}
```

Sparse similarity in Qdrant is always the dot product. There is no metric to choose, unlike the dense side where you pick Cosine, Dot, or Euclidean.

#### Sparse models: BM25, SPLADE, miniCOIL

| Model | How it assigns weights | Notes |
|-------|------------------------|-------|
| BM25 | Statistical: term frequency and inverse document frequency, no training | Classic, fast, interpretable. Scores tokens exactly as written. FastEmbed handle `Qdrant/bm25`. |
| SPLADE | Neural: a transformer expands text with related terms and weights them | Captures some synonymy while staying sparse. More compute than BM25. |
| miniCOIL | Neural, contextualized term weighting on BM25's exact vocabulary | Context aware exact match without full expansion cost. FastEmbed handle `Qdrant/minicoil-v1`. |

miniCOIL is Qdrant's recommendation for new projects. We use BM25 here because it needs no model inference at all, which keeps this notebook fast and every score traceable by hand.

#### How sparse is indexed

Qdrant uses an inverted index: for every token it keeps a posting list of every point where that token has a non-zero weight. A query only walks the posting lists for tokens it contains, skipping every point that shares none. HNSW (Module 2) is approximate, but the sparse index is exact.

### Key insight

Dense = meaning. Sparse = exact matching. Neither is complete alone. Every real query carries both semantic intent (what the user means) and exact constraints (what the user needs precisely).

## 3. Hybrid Search: Dense + Sparse

Hybrid search runs dense and sparse retrieval in the same request, then fuses the two ranked candidate lists into one result set: semantic understanding with exact-match precision. Payload filters constrain both retrievers while they run, which Section 7 covers.

**Reciprocal Rank Fusion (RRF)** merges the dense list and the sparse list using each candidate's *position* in the two lists, not its raw score. A document ranked high by both retrievers rises to the top. Because it ignores raw scores, RRF is robust to the fact that dense and sparse scores live on completely different scales.

## 4. Setting Up Hybrid Search in Qdrant

### Step 1: create a hybrid collection

Declare both a dense and a sparse vector config on one collection. Every point will carry both.

Two details in the cell below are easy to skip and expensive to skip. The sparse config needs `modifier=models.Modifier.IDF`, because BM25-style vectors store only term frequency and Qdrant applies the inverse-document-frequency half of the formula at query time. Without it you are not scoring BM25. And the payload index on `in_stock` is declared before any data is ingested, which is the ordering Module 4 explains.

In [ ]:
import warnings
from qdrant_client import QdrantClient, models

DENSE_MODEL = "sentence-transformers/all-MiniLM-L6-v2"  # 384-dim dense embeddings
SPARSE_MODEL = "Qdrant/bm25"                            # exact-token sparse

client = QdrantClient(":memory:")

client.create_collection(
    collection_name="products",
    vectors_config={
        "dense": models.VectorParams(size=384, distance=models.Distance.COSINE),
    },
    sparse_vectors_config={
        "sparse": models.SparseVectorParams(
            modifier=models.Modifier.IDF   # required for BM25 scoring
        ),
    },
)

with warnings.catch_warnings():
    warnings.simplefilter("ignore")   # local mode warns that indexes do nothing here
    client.create_payload_index(
        collection_name="products",
        field_name="in_stock",
        field_schema=models.PayloadSchemaType.BOOL,
    )

print("Hybrid collection 'products' created, with in_stock indexed.")

Hybrid collection 'products' created, with in_stock indexed.


### Step 2: insert points with both vectors

Each point carries a dense embedding and a sparse vector. We pass a `models.Document` and name the model. The client embeds it locally with FastEmbed before upload. The catalog below mixes a phone product line (to reproduce the `iPhone 15` problem) with running shoes (for the Nike example) and includes an exact SKU.

In [ ]:
catalog = [
    {"id": 1, "name": "iPhone 14",              "sku": "APL-IP14",    "price": 699, "in_stock": True},
    {"id": 2, "name": "iPhone 15",              "sku": "APL-IP15",    "price": 799, "in_stock": True},
    {"id": 3, "name": "iPhone 15 Pro Max",      "sku": "APL-IP15PM",  "price": 1199, "in_stock": True},
    {"id": 4, "name": "iPhone 13 mini",         "sku": "APL-IP13M",   "price": 599, "in_stock": False},
    {"id": 5, "name": "Nike Pegasus 40 running shoes", "sku": "NK-PEG40", "price": 130, "in_stock": True},
    {"id": 6, "name": "Nike Pegasus 39 running shoes", "sku": "NK-PEG39", "price": 110, "in_stock": True},
    {"id": 7, "name": "Adidas Ultraboost running shoes", "sku": "AD-UB22", "price": 180, "in_stock": True},
    {"id": 8, "name": "Widget assembly part SKU-48291", "sku": "SKU-48291", "price": 12, "in_stock": True},
]

client.upload_points(
    collection_name="products",
    points=[
        models.PointStruct(
            id=item["id"],
            vector={
                "dense":  models.Document(text=item["name"], model=DENSE_MODEL),
                "sparse": models.Document(text=item["name"] + " " + item["sku"], model=SPARSE_MODEL),
            },
            payload=item,
        )
        for item in catalog
    ],
)
print("Upserted", client.count("products").count, "products.")

Upserted 8 products.


### Step 3: three ways to search the same collection

A small helper prints results compactly. Then the same query runs three ways: dense-only, sparse-only, and hybrid.

Look closely at `hybrid()`. The filter is passed into **each** `Prefetch`, not to `query_points` as a top-level `query_filter`. That placement is the whole lesson of Section 7, and we break it on purpose later to show what goes wrong.

In [ ]:
def show(title, points):
    print(title)
    for r in points:
        print(f"  [{r.score:.4f}] {r.payload['name']}  (sku={r.payload['sku']}, in_stock={r.payload['in_stock']})")
    print()

def dense_only(text, query_filter=None, limit=5):
    return client.query_points(
        "products",
        query=models.Document(text=text, model=DENSE_MODEL),
        using="dense",
        query_filter=query_filter,   # no prefetch here, so top level is correct
        limit=limit,
    ).points

def sparse_only(text, query_filter=None, limit=5):
    return client.query_points(
        "products",
        query=models.Document(text=text, model=SPARSE_MODEL),
        using="sparse",
        query_filter=query_filter,   # no prefetch here either
        limit=limit,
    ).points

def hybrid(text, query_filter=None, limit=5, fusion="rrf"):
    # The filter goes INSIDE each prefetch, so both retrievers only ever
    # consider valid points. See Section 7 for why the top level is wrong here.
    fuse = (models.RrfQuery(rrf=models.Rrf()) if fusion == "rrf"
            else models.FusionQuery(fusion=models.Fusion.DBSF))
    return client.query_points(
        "products",
        prefetch=[
            models.Prefetch(query=models.Document(text=text, model=DENSE_MODEL),
                            using="dense", filter=query_filter, limit=20),
            models.Prefetch(query=models.Document(text=text, model=SPARSE_MODEL),
                            using="sparse", filter=query_filter, limit=20),
        ],
        query=fuse,
        limit=limit,
    ).points

The dense-only run reproduces the problem from Section 1: semantically similar phones cluster together, and the exact model the user typed does not reliably sit on top. Sparse-only, by contrast, locks onto the literal tokens.

In [ ]:
show("DENSE-ONLY  'iPhone 15'  (semantic neighborhood, exact model can drift):", dense_only("iPhone 15"))
show("SPARSE-ONLY 'iPhone 15'  (exact tokens win):", sparse_only("iPhone 15"))

DENSE-ONLY  'iPhone 15'  (semantic neighborhood, exact model can drift):
  [1.0000] iPhone 15  (sku=APL-IP15, in_stock=True)
  [0.8760] iPhone 14  (sku=APL-IP14, in_stock=True)
  [0.8149] iPhone 15 Pro Max  (sku=APL-IP15PM, in_stock=True)
  [0.6801] iPhone 13 mini  (sku=APL-IP13M, in_stock=False)
  [0.1764] Nike Pegasus 39 running shoes  (sku=NK-PEG39, in_stock=True)

SPARSE-ONLY 'iPhone 15'  (exact tokens win):
  [3.3050] iPhone 15  (sku=APL-IP15, in_stock=True)
  [3.2874] iPhone 15 Pro Max  (sku=APL-IP15PM, in_stock=True)
  [1.1605] iPhone 14  (sku=APL-IP14, in_stock=True)
  [1.1574] iPhone 13 mini  (sku=APL-IP13M, in_stock=False)



The clearest case is an exact code. A dense model has never really "seen" `SKU-48291` as a meaningful concept, so it drifts. Sparse matches the literal token exactly.

In [ ]:
show("DENSE-ONLY  'SKU-48291' (drifts):", dense_only("SKU-48291"))
show("SPARSE-ONLY 'SKU-48291' (exact hit):", sparse_only("SKU-48291"))

DENSE-ONLY  'SKU-48291' (drifts):
  [0.4544] Widget assembly part SKU-48291  (sku=SKU-48291, in_stock=True)
  [0.2397] Nike Pegasus 39 running shoes  (sku=NK-PEG39, in_stock=True)
  [0.1948] Nike Pegasus 40 running shoes  (sku=NK-PEG40, in_stock=True)
  [0.1403] Adidas Ultraboost running shoes  (sku=AD-UB22, in_stock=True)
  [0.1266] iPhone 15  (sku=APL-IP15, in_stock=True)

SPARSE-ONLY 'SKU-48291' (exact hit):
  [6.7829] Widget assembly part SKU-48291  (sku=SKU-48291, in_stock=True)



Now hybrid. Both prefetches run as part of one request and return up to 20 candidates each. RRF merges them by rank, then `limit` takes the top results. Because each prefetch carries the filter, out-of-stock products never enter either candidate set.

In [ ]:
from qdrant_client.models import Filter, FieldCondition, MatchValue

in_stock_filter = Filter(must=[FieldCondition(key="in_stock", match=MatchValue(value=True))])

show("HYBRID 'Nike Pegasus 40 size 10' (semantic + exact, in stock only):",
     hybrid("Nike Pegasus 40 size 10", query_filter=in_stock_filter))

HYBRID 'Nike Pegasus 40 size 10' (semantic + exact, in stock only):
  [1.0000] Nike Pegasus 40 running shoes  (sku=NK-PEG40, in_stock=True)
  [0.6667] Nike Pegasus 39 running shoes  (sku=NK-PEG39, in_stock=True)
  [0.2500] Adidas Ultraboost running shoes  (sku=AD-UB22, in_stock=True)
  [0.2000] iPhone 15  (sku=APL-IP15, in_stock=True)
  [0.1667] iPhone 14  (sku=APL-IP14, in_stock=True)



### Try it: watch the ranking change

The experiment from the lesson, side by side. Compare where the exact target lands under dense-only versus hybrid.

In [ ]:
q = "Nike Pegasus 40"
show(f"DENSE-ONLY  '{q}':", dense_only(q))
show(f"HYBRID      '{q}':", hybrid(q))

DENSE-ONLY  'Nike Pegasus 40':
  [0.8815] Nike Pegasus 39 running shoes  (sku=NK-PEG39, in_stock=True)
  [0.8713] Nike Pegasus 40 running shoes  (sku=NK-PEG40, in_stock=True)
  [0.6043] Adidas Ultraboost running shoes  (sku=AD-UB22, in_stock=True)
  [0.2403] iPhone 15  (sku=APL-IP15, in_stock=True)
  [0.2044] iPhone 14  (sku=APL-IP14, in_stock=True)



HYBRID      'Nike Pegasus 40':
  [0.8333] Nike Pegasus 39 running shoes  (sku=NK-PEG39, in_stock=True)
  [0.8333] Nike Pegasus 40 running shoes  (sku=NK-PEG40, in_stock=True)
  [0.2500] Adidas Ultraboost running shoes  (sku=AD-UB22, in_stock=True)
  [0.2000] iPhone 15  (sku=APL-IP15, in_stock=True)
  [0.1667] iPhone 14  (sku=APL-IP14, in_stock=True)



### Try it: put the filter in the wrong place

Now the part worth doing yourself, because nothing raises an error when you get it wrong.

`hybrid_wrong()` below is identical to `hybrid()` except the filter moves from inside the prefetches to a top-level `query_filter`. Then we mark `iPhone 15` out of stock and run both against an in-stock-only filter. The correct version drops it. The broken version should not.

In [ ]:
def hybrid_wrong(text, query_filter=None, limit=5):
    """Same query, filter at the top level instead of inside each prefetch."""
    return client.query_points(
        "products",
        prefetch=[
            models.Prefetch(query=models.Document(text=text, model=DENSE_MODEL),
                            using="dense", limit=20),
            models.Prefetch(query=models.Document(text=text, model=SPARSE_MODEL),
                            using="sparse", limit=20),
        ],
        query=models.RrfQuery(rrf=models.Rrf()),
        query_filter=query_filter,     # too late: prefetches have already run
        limit=limit,
    ).points

client.set_payload("products", payload={"in_stock": False}, points=[2])  # iPhone 15

show("CORRECT  filter inside each prefetch (iPhone 15 is gone):",
     hybrid("iPhone 15", query_filter=in_stock_filter))
show("BROKEN   filter at the top level (iPhone 15 comes back):",
     hybrid_wrong("iPhone 15", query_filter=in_stock_filter))

CORRECT  filter inside each prefetch (iPhone 15 is gone):
  [0.8333] iPhone 14  (sku=APL-IP14, in_stock=True)
  [0.8333] iPhone 15 Pro Max  (sku=APL-IP15PM, in_stock=True)
  [0.2500] Nike Pegasus 39 running shoes  (sku=NK-PEG39, in_stock=True)
  [0.2000] Nike Pegasus 40 running shoes  (sku=NK-PEG40, in_stock=True)
  [0.1667] Adidas Ultraboost running shoes  (sku=AD-UB22, in_stock=True)



BROKEN   filter at the top level (iPhone 15 comes back):
  [1.0000] iPhone 15  (sku=APL-IP15, in_stock=False)
  [0.5833] iPhone 14  (sku=APL-IP14, in_stock=True)
  [0.5833] iPhone 15 Pro Max  (sku=APL-IP15PM, in_stock=True)
  [0.4000] iPhone 13 mini  (sku=APL-IP13M, in_stock=False)
  [0.1667] Nike Pegasus 39 running shoes  (sku=NK-PEG39, in_stock=True)



There it is. Same filter, same query, one line of difference, and the broken version returns a product that is out of stock, without a warning or an error.

The reason is execution order. Once a query has prefetches, Qdrant runs those first and applies the main query to their results. A top-level filter therefore never reaches the retrievers: each one searches the whole catalog, and the filter only trims the already-fused list at the end. That is post-filtering, and with a selective filter it can leave you with nothing at all.

The rule is simple. No prefetch, use `query_filter`. Prefetch, put the filter in every prefetch.

In [ ]:
client.set_payload("products", payload={"in_stock": True}, points=[2])  # restore iPhone 15
print("iPhone 15 back in stock:", client.retrieve("products", ids=[2])[0].payload["in_stock"])

iPhone 15 back in stock: True


## 5. Fusion Strategies

Once both retrievers return candidates, a fusion algorithm merges them into one ranked list. Qdrant supports two.

| Strategy | How it works | When to use it |
|----------|--------------|----------------|
| RRF (Reciprocal Rank Fusion) | Combines rankings only, ignores raw scores. Robust, hard to game. | Default. Safe when dense and sparse score scales differ. |
| DBSF (Distribution-Based Score Fusion) | Normalizes score distributions before merging. Sensitive to relative score gaps. | When score gaps meaningfully encode relevance and both retrievers are well calibrated. |

Start with unweighted RRF, because dense and sparse scores live on different scales and raw-score fusion without normalization is unreliable. RRF also accepts a `k` constant and per-prefetch `weights`, so you can favour the stronger retriever once you have an evaluation set to tune against. Move to DBSF or tuned weights only after measuring, and tune on a different split from the one you measure on.

Both strategies run below on the same query so you can compare.

In [ ]:
show("RRF  fusion:",  hybrid("Nike Pegasus 40 size 10", fusion="rrf"))
show("DBSF fusion:", hybrid("Nike Pegasus 40 size 10", fusion="dbsf"))

RRF  fusion:
  [1.0000] Nike Pegasus 40 running shoes  (sku=NK-PEG40, in_stock=True)
  [0.6667] Nike Pegasus 39 running shoes  (sku=NK-PEG39, in_stock=True)
  [0.2500] Adidas Ultraboost running shoes  (sku=AD-UB22, in_stock=True)
  [0.2000] iPhone 15  (sku=APL-IP15, in_stock=True)
  [0.1667] iPhone 14  (sku=APL-IP14, in_stock=True)



DBSF fusion:
  [1.3562] Nike Pegasus 40 running shoes  (sku=NK-PEG40, in_stock=True)
  [1.1153] Nike Pegasus 39 running shoes  (sku=NK-PEG39, in_stock=True)
  [0.6049] Adidas Ultraboost running shoes  (sku=AD-UB22, in_stock=True)
  [0.4129] iPhone 15  (sku=APL-IP15, in_stock=True)
  [0.4019] iPhone 14  (sku=APL-IP14, in_stock=True)



## 6. Beyond Text: Multimodal Search

The same primitive, embed data then store as a vector then search by similarity, applies to any modality: images (CLIP, SigLIP), video frames, audio fingerprints, or text. Qdrant stores whatever vectors your embedding model produces, and the retrieval mechanics are identical.

**Data to Embedding Model to Vector to Qdrant.** The modality changes, the system does not.

### Named vectors

When two representations must be searchable together, store them as named vectors on the same point, then query against whichever one you want. Below we demonstrate the mechanic with two text views of each product, a short `title` and a longer `description`, so it runs with no heavy image models. In a real multimodal system you would swap the `description` model for an image encoder such as CLIP; the collection and query code stay the same shape.

In [ ]:
client.create_collection(
    collection_name="catalog",
    vectors_config={
        "title":       models.VectorParams(size=384, distance=models.Distance.COSINE),
        "description": models.VectorParams(size=384, distance=models.Distance.COSINE),
    },
)

client.upload_points(
    collection_name="catalog",
    points=[
        models.PointStruct(
            id=42,
            vector={
                "title":       models.Document(text="Red Nike running shoe", model=DENSE_MODEL),
                "description": models.Document(text="Lightweight breathable trainer for road running in bright red", model=DENSE_MODEL),
            },
            payload={"sku": "NK-RED-10", "price": 120},
        ),
        models.PointStruct(
            id=43,
            vector={
                "title":       models.Document(text="Blue hiking boot", model=DENSE_MODEL),
                "description": models.Document(text="Waterproof ankle support boot for rough mountain trails", model=DENSE_MODEL),
            },
            payload={"sku": "HK-BLU-9", "price": 150},
        ),
    ],
)

# Query one named vector; the other is simply not searched on this call.
res = client.query_points(
    "catalog",
    query=models.Document(text="shoe for running on pavement", model=DENSE_MODEL),
    using="description",
    limit=2,
).points
for r in res:
    print(f"[{r.score:.4f}] {r.payload['sku']}")

[0.5492] HK-BLU-9
[0.4119] NK-RED-10


Each named vector is its own space, and vectors from different models are not comparable. Swapping `description` for a CLIP image encoder would look like this, using the same collection and query shape:

```python
# ingestion
"image": embed_image(product_photo)          # CLIP get_image_features

# querying by text against those image vectors
client.query_points("catalog", query=embed_text_with_clip("red running shoe"),
                   using="image", limit=10)
```

The important part is that the query text must go through **CLIP's text encoder**, not the sentence transformer, or it lands in a different space and the scores are meaningless. Module 5 builds this out properly.

## 7. Filtering Works with Any Retrieval Method

Payload filters are not a hybrid-only feature. The same conditions apply to dense-only, sparse-only, or hybrid retrieval, and they are evaluated as hard constraints *while* the search runs, not as a separate step afterward. Because out-of-scope points never take a slot in your top-K, results stay both relevant and valid: in stock, within permissions, within a date range.

What changes between the three is *where* the filter goes, which is what the broken example earlier demonstrated. Dense-only and sparse-only have no prefetch, so `query_filter` is correct. Hybrid has prefetches, so the filter belongs in each one.

The three calls below apply the identical filter to all three retrieval methods.

In [ ]:
show("DENSE + filter:",  dense_only("iPhone", query_filter=in_stock_filter))
show("SPARSE + filter:", sparse_only("iPhone", query_filter=in_stock_filter))
show("HYBRID + filter:", hybrid("iPhone", query_filter=in_stock_filter))

DENSE + filter:
  [0.8036] iPhone 15  (sku=APL-IP15, in_stock=True)
  [0.7914] iPhone 14  (sku=APL-IP14, in_stock=True)
  [0.5714] iPhone 15 Pro Max  (sku=APL-IP15PM, in_stock=True)
  [0.1663] Nike Pegasus 39 running shoes  (sku=NK-PEG39, in_stock=True)
  [0.1396] Adidas Ultraboost running shoes  (sku=AD-UB22, in_stock=True)

SPARSE + filter:
  [1.1605] iPhone 15  (sku=APL-IP15, in_stock=True)
  [1.1605] iPhone 14  (sku=APL-IP14, in_stock=True)
  [1.1543] iPhone 15 Pro Max  (sku=APL-IP15PM, in_stock=True)

HYBRID + filter:
  [1.0000] iPhone 15  (sku=APL-IP15, in_stock=True)
  [0.6667] iPhone 14  (sku=APL-IP14, in_stock=True)
  [0.5000] iPhone 15 Pro Max  (sku=APL-IP15PM, in_stock=True)
  [0.2000] Nike Pegasus 39 running shoes  (sku=NK-PEG39, in_stock=True)
  [0.1667] Adidas Ultraboost running shoes  (sku=AD-UB22, in_stock=True)



## What's next: Module 4

- The five layers of a vector search stack
- A worked design: a multilingual news search system, decision by decision
- Filtering in production: how the query planner picks a strategy, and multitenancy
- The production RAG pipeline, and deployment options

[Continue to Module 4](https://qdrant.tech/course/beginners/module-4/)